# Basics of lancedb

In [25]:
import lancedb 

db = lancedb.connect("knowledge_base")
db

LanceDBConnection(uri='c:\\Users\\adelo\\de25\\ai_ops_lecture\\09_lancedb\\knowledge_base')

In [26]:
db.uri

'c:\\Users\\adelo\\de25\\ai_ops_lecture\\09_lancedb\\knowledge_base'

## Create table

In [27]:
import json

with open("animals_text_embeddings.json", encoding="utf-8") as f:
    data = json.load(f)

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [28]:
data[0].get("text")

'A small brown dog running.'

In [29]:
data[0].get("vector")

[0.12, 0.85, 0.33]

In [30]:

db.create_table("animals_text", exist_ok=True, data=data)
db["animals_text"]

LanceTable(name='animals_text', _conn=LanceDBConnection(uri='c:\\Users\\adelo\\de25\\ai_ops_lecture\\09_lancedb\\knowledge_base'))

In [31]:
db["animals_text"].head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

convert to dataframe

In [32]:
db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [33]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]
db["animals_text"].add(more_data)

AddResult(version=4)

In [34]:
db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Drop a table

In [35]:
db

LanceDBConnection(uri='c:\\Users\\adelo\\de25\\ai_ops_lecture\\09_lancedb\\knowledge_base')

In [36]:
from lancedb.pydantic import LanceModel

class CoolSchema(LanceModel):
    cool_name: str 
    cool_number: int = 42

# mode overwrite -> replace in SQL 
# exist_ok = True -> if not exists in SQL
db.create_table("cool_table", schema=CoolSchema, mode="overwrite")

LanceTable(name='cool_table', _conn=LanceDBConnection(uri='c:\\Users\\adelo\\de25\\ai_ops_lecture\\09_lancedb\\knowledge_base'))

In [37]:
import pandas as pd 
cool_data = [{"cool_name": "pi", "cool_number": 3.141592}, {"cool_name": "e", "cool_number": 2.713}]
cool_df = pd.DataFrame(cool_data)
cool_df

,cool_name,cool_number
0,pi,3.141592
1,e,2.713000


In [38]:
db["cool_table"].add(cool_df)

AddResult(version=2)

In [39]:
db["cool_table"].to_pandas()

,cool_name,cool_number
0,pi,3
1,e,2


In [40]:
db.drop_table("cool_table")

In [41]:
db.list_tables().tables

['animals_text', 'jokes', 'jokes_nvidia', 'jokes_sentence_transformer']

## Vector search

In [42]:
db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [43]:
# common flow in many vector databases
# natural language -> embedder -> query_vector
query_vector = [0.5, 0.2, 0.9]
db["animals_text"].search(query=query_vector).limit(3).to_pandas()

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
2,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354


In [44]:
print("Want to do this: ask a question directly")
try:
    db["animals_text"].search(query="what is a pandas eating").limit(3).to_pandas()
except ValueError as err:
    print(err)

Want to do this: ask a question directly
Invalid input, Cannot perform full text search unless an INVERTED index has been created on at least one column


## Embeddings API

In [45]:
from lancedb.pydantic import Vector
from lancedb.embeddings import get_registry
from dotenv import load_dotenv 

load_dotenv()

# NOTE: in .env the key should be called: COHERE_API_KEY
model = get_registry().get("cohere").create(name="embed-multilingual-v3.0")

class JokeModel(LanceModel):
    jokes: str = model.SourceField() 
    vector: Vector(model.ndims()) = model.VectorField()  

print(f"{model.ndims() = }")

db.create_table("jokes", schema=JokeModel, exist_ok=True)


model.ndims() = 1024


LanceTable(name='jokes', _conn=LanceDBConnection(uri='c:\\Users\\adelo\\de25\\ai_ops_lecture\\09_lancedb\\knowledge_base'))

In [47]:
import json

with open("jokes.json", encoding="utf-8") as f:
    data = json.load(f)

data

{'jokes': ['Parallel lines have so much in common—it’s sad they’ll never meet.',
  'ETL stands for “Extract, Transform, Leave for the next person.”',
  'What do you call a snake that runs your scripts? A py-thon (ton of errors).',
  'Gold walks into a bar. The bartender says, “Au, get out!”',
  'C# devs don’t argue; they just throw exceptions.',
  'I asked the data lake if it had my file. It said: “Sure… somewhere.”',
  'My math teacher said I’m average… how mean!',
  'Python is so friendly even whitespace gets a say.',
  'The C# compiler walked into a bar. The bartender said, “You’re not allowed here!” It replied, “But I’ve got excellent syntax!”',
  'Never trust an atom—they make up everything.',
  'Why was the equal sign so humble? It knew it wasn’t < or > than anyone.',
  'My Python code works on the first try… said no one ever.',
  'Why did the chemist ground his kids? Because they were being basic.',
  "Our data pipeline broke… but don't worry, it was by design. We call it a feat

In [ ]:
import pandas as pd

# jokes.json has the shape {"jokes": [...]} -> one column called "jokes",
# which matches the SourceField in JokeModel. The vector column is
# filled in automatically by the Cohere embedding function.
df = pd.DataFrame(data)

# mode="overwrite" keeps the cell re-runnable (no duplicate rows)
db["jokes"].add(df, mode="overwrite")


In [ ]:
db["jokes"].to_pandas().head()

,jokes,vector
0,Parallel lines have so much in common—it’s sad...,"[-0.00301, 0.001863, -0.014427, 0.051575, -0.0..."
1,"ETL stands for “Extract, Transform, Leave for ...","[0.01239, 0.0047, -0.091431, 0.080627, -0.0505..."
2,What do you call a snake that runs your script...,"[0.019012, 0.044525, -0.021454, 0.02887, -0.00..."
3,"Gold walks into a bar. The bartender says, “Au...","[0.019958, 0.060089, 0.012733, 0.024048, -0.07..."
4,C# devs don’t argue; they just throw exceptions.,"[0.013275, 0.002462, -0.057007, 0.060883, -0.0..."


In [ ]:
db["jokes"].search("ge mig kemiskämt").limit(4).to_pandas()

,jokes,vector,_distance
0,Oxygen and Magnesium got together—OMg!,"[-0.041138, -0.004559, 0.007397, 0.006527, -0....",1.150106
1,I told a chemistry joke… there was no reaction.,"[-0.032745, 0.018295, 0.039368, 0.001193, -0.0...",1.175761
2,Chemists have all the solutions… mostly in bea...,"[0.00139, 0.030945, -0.001131, 0.021011, -0.05...",1.231410
3,Never trust an atom—they make up everything.,"[-0.009163, 0.003523, -0.020401, 0.049408, -0....",1.269136


In [ ]:
df.iloc[0].jokes

'Parallel lines have so much in common—it’s sad they’ll never meet.'

In [ ]:
db["jokes"].search(df.iloc[0].jokes).limit(4).to_pandas()

,jokes,vector,_distance
0,Parallel lines have so much in common—it’s sad...,"[-0.00301, 0.001863, -0.014427, 0.051575, -0.0...",0.192160
1,"Dear algebra, please stop asking us to find yo...","[0.011345, 0.004944, -0.019363, 0.060486, -0.0...",1.187968
2,I tried to explain async/await to my friend… n...,"[0.002825, 0.007874, -0.016235, 0.005787, -0.0...",1.231241
3,My Python code works on the first try… said no...,"[0.020096, 0.035461, -0.008324, 0.048584, -0.0...",1.274799


In [ ]:
db["jokes"].search("varför är matteboken ledsen?").limit(4).to_pandas()

,jokes,vector,_distance
0,My math teacher said I’m average… how mean!,"[-0.002581, 0.031525, 0.013107, 0.030991, -0.0...",1.098703
1,Why’s 6 afraid of 7? Because 7 8 9.,"[0.005367, 0.020432, -0.002979, 0.016785, -0.0...",1.146541
2,"Dear algebra, please stop asking us to find yo...","[0.011345, 0.004944, -0.019363, 0.060486, -0.0...",1.152071
3,Why do data engineers hate nature? Too many un...,"[0.038971, 0.061859, -0.02388, 0.050293, -0.03...",1.287978
